# 02. Combinatorics & LLM Sampling Strategies

**The mathematics of discrete probability, temperature scaling, Top-k, Top-p (Nucleus) sampling, and Beam Search in Large Language Models.**

---

## 1. Combinatorics Fundamentals

- **Permutations (Order matters)**: $P(n, k) = \frac{n!}{(n-k)!}$
- **Combinations (Order does not matter)**: $\binom{n}{k} = \frac{n!}{k!(n-k)!}$
- **Multinomial Coefficient**: $\binom{n}{k_1, k_2, \dots, k_m} = \frac{n!}{k_1! k_2! \dots k_m!}$

---

## 2. LLM Autoregressive Generation & Logits

An autoregressive Large Language Model (e.g. GPT-4, LLaMA) generates text token by token. At each step, given prompt tokens $w_{1:t-1}$, the final linear layer produces a vector of unnormalized scores called **Logits** $\mathbf{z} \in \mathbb{R}^{|V|}$ over the vocabulary $V$ (where $|V| \approx 32,000 - 128,000$).

---

## 3. Temperature Scaling

To convert logits into a probability distribution, we apply Softmax with a **Temperature parameter $T > 0$**:

$$P(w_i) = \frac{\exp(z_i / T)}{\sum_{j=1}^{|V|} \exp(z_j / T)}$$

- **$T = 1.0$**: Standard unscaled Softmax probabilities.
- **$T \to 0$ (Cold)**: Approximates an $\arg\max$ one-hot distribution (**Greedy Decoding**). Output becomes strictly deterministic, repetitive, and factual.
- **$T > 1$ (Hot)**: Flattens the distribution towards uniform randomness. Output becomes creative and diverse, but risks hallucination/incoherence.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualizing Temperature Scaling on Vocabulary Logits
vocab = ['the', 'cat', 'sat', 'on', 'quantum', 'banana']
logits = np.array([4.0, 3.2, 2.8, 2.1, 0.5, -0.2])

def softmax_temperature(logits, T):
    scaled_logits = logits / T
    exp_l = np.exp(scaled_logits - np.max(scaled_logits))
    return exp_l / exp_l.sum()

p_cold = softmax_temperature(logits, T=0.3)
p_norm = softmax_temperature(logits, T=1.0)
p_hot  = softmax_temperature(logits, T=2.5)

x = np.arange(len(vocab))
width = 0.25

plt.figure(figsize=(10, 5))
plt.bar(x - width, p_cold, width, label='Cold (T=0.3) - Deterministic', color='blue', alpha=0.8)
plt.bar(x, p_norm, width, label='Standard (T=1.0) - Balanced', color='green', alpha=0.8)
plt.bar(x + width, p_hot, width, label='Hot (T=2.5) - Creative/Random', color='orange', alpha=0.8)
plt.xticks(x, vocab)
plt.title("Effect of Temperature Scaling on LLM Next-Token Probabilities")
plt.ylabel("Probability P(w)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


---

## 4. Top-$k$ and Top-$p$ (Nucleus) Sampling

### 4.1 Top-$k$ Sampling (Fan et al., 2018):
Sort vocabulary tokens by probability and truncate to only the **top $k$ most likely tokens**, setting all other probabilities to 0 and renormalizing:
$$V^{(k)} = \text{top } k \text{ tokens}, \quad P'(w_i) = \begin{cases} \frac{P(w_i)}{\sum_{w \in V^{(k)}} P(w)} & \text{if } w_i \in V^{(k)} \\ 0 & \text{otherwise} \end{cases}$$

### 4.2 Top-$p$ (Nucleus) Sampling (Holtzman et al., 2019):
Instead of a fixed $k$, dynamically choose the smallest subset of tokens $V^{(p)}$ whose **cumulative probability exceeds threshold $p$** (e.g. $p = 0.9$):
$$\sum_{w \in V^{(p)}} P(w) \geq p$$
- When the model is confident, $V^{(p)}$ may contain just 1 or 2 tokens.
- When the model is uncertain, $V^{(p)}$ expands dynamically!


In [ ]:
def top_k_sampling(probs, k):
    sorted_indices = np.argsort(probs)[::-1]
    top_k_idx = sorted_indices[:k]
    
    new_probs = np.zeros_like(probs)
    new_probs[top_k_idx] = probs[top_k_idx]
    return new_probs / new_probs.sum()

def top_p_sampling(probs, p):
    sorted_indices = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_indices]
    
    cumulative_probs = np.cumsum(sorted_probs)
    # Cutoff at threshold p
    cutoff_idx = np.where(cumulative_probs > p)[0]
    last_idx = cutoff_idx[0] + 1 if len(cutoff_idx) > 0 else len(probs)
    
    selected_indices = sorted_indices[:last_idx]
    new_probs = np.zeros_like(probs)
    new_probs[selected_indices] = probs[selected_indices]
    return new_probs / new_probs.sum()

# Test Top-k and Top-p
p_base = softmax_temperature(logits, T=1.0)
p_top_k = top_k_sampling(p_base, k=3)
p_top_p = top_p_sampling(p_base, p=0.85)

print("Original Vocab Probabilities:\n", dict(zip(vocab, np.round(p_base, 3))))
print("\nTop-k (k=3) Filtered:\n", dict(zip(vocab, np.round(p_top_k, 3))))
print("\nTop-p (p=0.85 Nucleus) Filtered:\n", dict(zip(vocab, np.round(p_top_p, 3))))


---

## 5. Beam Search Decoding

### Greedy vs Exhaustive Search:
- **Greedy Search**: Selects $\arg\max P(w_t \mid w_{<t})$ at every step. Fast, but myopic (can get stuck in bad sentence trajectories).
- **Exhaustive Search**: Explores all $|V|^T$ possible sequences (combinatorially impossible, e.g. $50000^{100} \approx \infty$).

### Beam Search:
Maintains the top $B$ most probable candidate sequences (called the **Beam Width** $B \approx 3 - 10$) at each step:

$$\text{Score}(w_{1:t}) = \sum_{i=1}^t \log P(w_i \mid w_{<i})$$


---

## 6. Summary & Key Takeaways

1. **Temperature Scaling** $P(w) \propto \exp(z/T)$ controls exploration vs exploitation in LLMs.
2. **Top-$k$ Sampling** truncates to a fixed number of candidate tokens.
3. **Top-$p$ (Nucleus) Sampling** dynamically truncates the tail based on cumulative probability mass.
4. **Beam Search** balances greedy and exhaustive decoding for structured sequence generation.
